In [ ]:
# Fix numpy binary incompatibility (Kaggle environment)
# Run this cell first, then restart the kernel and run all cells again.
import subprocess, importlib
result = subprocess.run(["pip", "install", "-q", "--upgrade", "numpy"], capture_output=True, text=True)
print(result.stdout or "numpy upgraded")
print(">>> Restart the kernel now, then run all cells <<<")

In [ ]:
!pip install -q detectors datasets compressai

In [ ]:
import os, random, math
from io import BytesIO
from dataclasses import dataclass
from typing import List, Optional, Callable, Dict
from abc import ABC, abstractmethod

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
from torchvision.datasets import CIFAR10

import timm
import detectors  

from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Dataset and normalisation helpers (CIFAR-10)

In [ ]:
DATA_MEAN = (0.4914, 0.4822, 0.4465)
DATA_STD  = (0.2023, 0.1994, 0.2010)

DATA_MEAN_T = torch.tensor(DATA_MEAN, device=device).view(1, 3, 1, 1)
DATA_STD_T  = torch.tensor(DATA_STD,  device=device).view(1, 3, 1, 1)

def denormalize(x: torch.Tensor) -> torch.Tensor:
    return torch.clamp(x * DATA_STD_T + DATA_MEAN_T, 0.0, 1.0)

def renormalize(x: torch.Tensor) -> torch.Tensor:
    return (x - DATA_MEAN_T) / DATA_STD_T

def project_linf_pixel(x_adv: torch.Tensor, x0: torch.Tensor, eps: float) -> torch.Tensor:
    return torch.clamp(torch.max(torch.min(x_adv, x0 + eps), x0 - eps), 0.0, 1.0)

def batch_metrics_from_normalized(orig_norm, pert_norm):
    x0 = denormalize(orig_norm)
    x1 = denormalize(pert_norm)
    mse  = torch.mean((x0 - x1) ** 2, dim=(1, 2, 3))
    mae  = torch.mean(torch.abs(x0 - x1), dim=(1, 2, 3))
    psnr = 10.0 * torch.log10(1.0 / torch.clamp(mse, min=1e-10))
    return mse, mae, psnr

transform_test = T.Compose([T.ToTensor(), T.Normalize(DATA_MEAN, DATA_STD)])
cifar10_test = CIFAR10(root="./data", train=False, download=True, transform=transform_test)
loader = DataLoader(cifar10_test, batch_size=64, shuffle=False, num_workers=2)

images_norm, labels = next(iter(loader))
images_norm = images_norm.to(device)
labels = labels.to(device)
print(f"Batch shape: {images_norm.shape}, labels: {labels.shape}")

## Perturbation base classes

In [ ]:
class Perturbation(ABC):
    def __init__(self, name: str):
        self.name = name

    @abstractmethod
    def apply(self, model, images, labels, device) -> torch.Tensor:
        pass

@dataclass
class PerturbationPipeline:
    name: str
    steps: List[Perturbation]

    def apply(self, model, images, labels, device) -> torch.Tensor:
        x = images
        for step in self.steps:
            x = step.apply(model, x, labels, device)
        return x

## Compression implementations

In [ ]:
# ---- JPEG ----
class JpegCompressionPIL:
    def __init__(self, quality: int):
        self.quality = int(quality)

    def __call__(self, img):
        buf = BytesIO()
        img.save(buf, format="JPEG", quality=self.quality)
        buf.seek(0)
        return Image.open(buf).convert("RGB")


class JpegPerturbation(Perturbation):
    def __init__(self, quality: int):
        super().__init__(name=f"jpeg_q{quality}")
        self.quality = int(quality)
        self.jpeg = JpegCompressionPIL(quality)
        self.to_pil = T.ToPILImage()
        self.to_tensor = T.ToTensor()

    def apply(self, model, images, labels, device):
        pixels = denormalize(images.detach().to(device)).cpu()
        out = torch.stack([self.to_tensor(self.jpeg(self.to_pil(img))) for img in pixels]).to(device)
        return renormalize(out)


# ---- JPEG2000 ----
class Jpeg2000CompressionPIL:
    def __init__(self, quality: float, irreversible: bool = True):
        self.quality = float(quality)
        self.irreversible = irreversible

    @staticmethod
    def quality_to_rate(q: float) -> float:
        q = float(max(1.0, min(100.0, q)))
        lo, hi = 1.0, 100.0
        t = (100.0 - q) / 99.0
        return lo * ((hi / lo) ** t)

    def __call__(self, img):
        rate = self.quality_to_rate(self.quality)
        kwargs = {"quality_mode": "rates", "quality_layers": [rate], "irreversible": self.irreversible}
        last_exc = None
        for fmt in ("JPEG2000", "JP2"):
            try:
                with BytesIO() as buf:
                    img.save(buf, format=fmt, **kwargs)
                    buf.seek(0)
                    out = Image.open(buf).convert("RGB")
                    out.load()
                    return out
            except Exception as exc:
                last_exc = exc
        raise RuntimeError("JPEG2000 not available") from last_exc


class Jpeg2000Perturbation(Perturbation):
    def __init__(self, quality: float):
        super().__init__(name=f"jpeg2000_q{int(quality)}")
        self.jpeg2000 = Jpeg2000CompressionPIL(quality)
        self.to_pil = T.ToPILImage()
        self.to_tensor = T.ToTensor()

    def apply(self, model, images, labels, device):
        pixels = denormalize(images.detach().to(device)).cpu()
        out = torch.stack([self.to_tensor(self.jpeg2000(self.to_pil(img))) for img in pixels]).to(device)
        return renormalize(torch.clamp(out, 0.0, 1.0))


# ---- PCA ----
class PcaPerturbation(Perturbation):
    def __init__(self, quality: float):
        super().__init__(name=f"pca_q{int(quality)}")
        self.quality = float(quality)

    def apply(self, model, images, labels, device):
        x = denormalize(images.detach().to(device))
        B, C, H, W = x.shape
        k = max(1, int(round(self.quality / 100.0 * min(H, W))))
        x_flat = x.view(B * C, H, W)
        x_out = torch.empty_like(x_flat)
        for i in range(x_flat.size(0)):
            A = x_flat[i]
            row_mean = A.mean(dim=1, keepdim=True)
            U, S, Vh = torch.linalg.svd(A - row_mean, full_matrices=False)
            x_out[i] = (U[:, :k] * S[:k]) @ Vh[:k, :] + row_mean
        return renormalize(torch.clamp(x_out.view(B, C, H, W), 0.0, 1.0))


# ---- PatchSVD ----
class PatchSVDPerturbation(Perturbation):
    def __init__(self, quality: float, patch_size: int = 8):
        super().__init__(name=f"patchsvd_q{int(quality)}_p{patch_size}")
        self.quality = float(quality)
        self.patch_size = int(patch_size)

    def apply(self, model, images, labels, device):
        x = denormalize(images.detach().to(device))
        B, C, H, W = x.shape
        p = self.patch_size
        pad_h = (p - H % p) % p
        pad_w = (p - W % p) % p
        if pad_h or pad_w:
            x = F.pad(x, (0, pad_w, 0, pad_h))
        H_pad, W_pad = x.shape[2], x.shape[3]
        patches = x.unfold(2, p, p).unfold(3, p, p)
        GH, GW = patches.shape[2], patches.shape[3]
        patches_flat = patches.contiguous().view(-1, p, p)
        U, S, Vh = torch.linalg.svd(patches_flat, full_matrices=False)
        k = max(1, int(round(self.quality / 100.0 * p)))
        recon_flat = (U[:, :, :k] * S[:, :k].unsqueeze(1)) @ Vh[:, :k, :]
        recon = recon_flat.view(B, C, GH, GW, p, p).permute(0, 1, 2, 4, 3, 5).contiguous().view(B, C, H_pad, W_pad)
        if pad_h or pad_w:
            recon = recon[:, :, :H, :W]
        return renormalize(torch.clamp(recon, 0.0, 1.0))

## Attack implementations

In [ ]:
class FgsmPerturbation(Perturbation):
    def __init__(self, epsilon: float):
        super().__init__(name=f"fgsm_eps{epsilon}")
        self.epsilon = float(epsilon)

    def apply(self, model, images, labels, device):
        model.eval()
        x0_px = denormalize(images.to(device)).detach()
        x_adv = x0_px.clone().requires_grad_(True)
        loss = F.cross_entropy(model(renormalize(x_adv)), labels.to(device))
        grad = torch.autograd.grad(loss, x_adv)[0]
        x_adv = project_linf_pixel(x0_px + self.epsilon * grad.sign(), x0_px, self.epsilon)
        return renormalize(x_adv.detach())


class PGDPerturbation(Perturbation):
    def __init__(self, epsilon: float, steps: int = 10, alpha: float = None, random_start: bool = True):
        self.epsilon = float(epsilon)
        self.steps = int(steps)
        self.alpha = float(alpha) if alpha is not None else 2.0 / 255.0
        self.random_start = bool(random_start)
        super().__init__(name=f"pgd_eps{self.epsilon}_k{self.steps}_rs{int(self.random_start)}")

    def apply(self, model, images, labels, device):
        model.eval()
        x0_px = denormalize(images.to(device)).detach()
        if self.random_start:
            noise = torch.empty_like(x0_px).uniform_(-self.epsilon, self.epsilon)
            x_adv = project_linf_pixel(x0_px + noise, x0_px, self.epsilon).detach()
        else:
            x_adv = x0_px.clone().detach()
        for _ in range(self.steps):
            x_adv = x_adv.requires_grad_(True)
            grad = torch.autograd.grad(F.cross_entropy(model(renormalize(x_adv)), labels.to(device)), x_adv)[0]
            with torch.no_grad():
                x_adv = project_linf_pixel(x_adv + self.alpha * grad.sign(), x0_px, self.epsilon)
        return renormalize(x_adv.detach())


def dlr_loss(logits, labels):
    sorted_logits, indices = torch.sort(logits, dim=1, descending=True)
    z_y     = logits[torch.arange(logits.shape[0], device=logits.device), labels]
    z_max1  = sorted_logits[:, 0]
    z_other = torch.where(indices[:, 0] == labels, sorted_logits[:, 1], z_max1)
    return -(z_y - z_other) / (z_max1 - sorted_logits[:, 2] + 1e-12)


class APGDPerturbation(Perturbation):
    def __init__(self, epsilon=8/255, num_steps=20, num_restarts=1, loss_type="dlr", step_factor=2.0):
        super().__init__(name=f"apgd_{loss_type}_eps{epsilon}")
        self.epsilon = float(epsilon)
        self.num_steps = int(num_steps)
        self.num_restarts = int(num_restarts)
        self.loss_type = str(loss_type).lower()
        self.step_factor = float(step_factor)

    def apply(self, model, images, labels, device):
        model.eval()
        x0_px = denormalize(images.detach().to(device)).detach()
        y = labels.to(device)
        x_best = x0_px.clone()
        loss_best = torch.full((x0_px.size(0),), -1e10, device=device)
        c1 = max(1, int(0.22 * self.num_steps))
        c2 = max(1, int(0.75 * self.num_steps))
        checkpoints = {c1, c2}
        for _ in range(self.num_restarts):
            x = project_linf_pixel(x0_px + torch.empty_like(x0_px).uniform_(-self.epsilon, self.epsilon), x0_px, self.epsilon)
            eta = self.step_factor * self.epsilon
            for i in range(self.num_steps):
                x = x.detach().requires_grad_(True)
                logits = model(renormalize(x))
                loss_indiv = F.cross_entropy(logits, y, reduction='none') if self.loss_type == "ce" else dlr_loss(logits, y)
                grad = torch.autograd.grad(loss_indiv.sum(), x)[0]
                with torch.no_grad():
                    improved = loss_indiv > loss_best
                    loss_best[improved] = loss_indiv[improved]
                    x_best[improved] = x[improved].detach()
                    x = project_linf_pixel(x + eta * grad.sign(), x0_px, self.epsilon)
                    if (i + 1) in checkpoints:
                        eta *= 0.5
        return renormalize(x_best.detach())

## Model loader

In [ ]:
def load_model(name: str) -> nn.Module:
    model = timm.create_model(name, pretrained=True)
    return model.to(device).eval()

model = load_model("resnet18_cifar10")
print("Model loaded:", type(model).__name__)

---
## Check 1 — Compression high-quality reconstruction (JPEG / JPEG2000 at quality=100)

Expected: PSNR > 35 dB and 50 dB, respectively, indicating high-quality reconstruction.

In [ ]:
print("=" * 60)
print("CHECK 1a: JPEG and JPEG2000 at quality=100 → high PSNR")
print("=" * 60)
print("Note: JPEG is lossy by design (DCT quantisation) even at q=100.")
print("      Typical PSNR: JPEG ~35-45 dB, JPEG2000 ~50+ dB.")
print()

THRESHOLDS = {"JPEG q=100": 35, "JPEG2000 q=100": 50}

for name, pert in [("JPEG q=100",     JpegPerturbation(quality=100)),
                   ("JPEG2000 q=100", Jpeg2000Perturbation(quality=100))]:
    out = pert.apply(None, images_norm, labels, device)
    _, _, psnr = batch_metrics_from_normalized(images_norm, out)
    mean_psnr = psnr.mean().item()
    threshold = THRESHOLDS[name]
    status = "PASS" if mean_psnr > threshold else "FAIL"
    print(f"  {name}: mean PSNR = {mean_psnr:.2f} dB  (threshold > {threshold} dB)  [{status}]")

## Check 2 — PCA / PatchSVD at 100% singular value retention → lossless

Expected: reconstruction error within floating-point precision (PSNR > 60 dB, MSE ≈ 0).

In [ ]:
print("=" * 60)
print("CHECK 2: PCA / PatchSVD at quality=100 → near-exact reconstruction")
print("=" * 60)

for name, pert in [("PCA q=100",      PcaPerturbation(quality=100)),
                   ("PatchSVD q=100", PatchSVDPerturbation(quality=100, patch_size=8))]:
    out = pert.apply(None, images_norm, labels, device)
    mse, _, psnr = batch_metrics_from_normalized(images_norm, out)
    mean_psnr = psnr.mean().item()
    mean_mse  = mse.mean().item()
    status = "PASS" if mean_psnr > 60 else "FAIL"
    print(f"  {name}: mean PSNR = {mean_psnr:.2f} dB, mean MSE = {mean_mse:.2e}  [{status}]")

## Check 3 — Attacks at ε=0 → output identical to input

Expected: PSNR = inf (or very large), MSE = 0.

In [ ]:
print("=" * 60)
print("CHECK 3: Attacks at ε=0 → output identical to input")
print("=" * 60)

for name, pert in [
    ("FGSM ε=0",  FgsmPerturbation(epsilon=0.0)),
    ("PGD ε=0",   PGDPerturbation(epsilon=0.0, steps=10, random_start=False)),
    ("APGD ε=0",  APGDPerturbation(epsilon=0.0, num_steps=5)),
]:
    with torch.enable_grad():
        out = pert.apply(model, images_norm, labels, device)
    mse, _, psnr = batch_metrics_from_normalized(images_norm, out)
    mean_mse = mse.mean().item()
    mean_psnr = psnr.mean().item()
    status = "PASS" if mean_mse < 1e-8 else "FAIL"
    print(f"  {name}: mean MSE = {mean_mse:.2e}, mean PSNR = {mean_psnr:.2f} dB  [{status}]")

## Check 4 — FGSM == PGD(steps=1, no random start)

Expected: the two outputs are identical (max absolute difference ≈ 0).

In [ ]:
print("=" * 60)
print("CHECK 4: FGSM == PGD(steps=1, no random start, α=ε)")
print("=" * 60)

EPS = 8 / 255

fgsm = FgsmPerturbation(epsilon=EPS)
pgd1 = PGDPerturbation(epsilon=EPS, steps=1, alpha=EPS, random_start=False)

with torch.enable_grad():
    out_fgsm = fgsm.apply(model, images_norm, labels, device)
    out_pgd1 = pgd1.apply(model, images_norm, labels, device)

max_diff = (out_fgsm - out_pgd1).abs().max().item()
status = "PASS" if max_diff < 1e-6 else "FAIL"
print(f"  Max absolute difference between outputs: {max_diff:.2e}  [{status}]")

## Check 5 — APGD accuracy decreases with increasing step count

Expected: accuracy is monotonically non-increasing as steps increase (stronger attack → lower accuracy).

In [ ]:
print("=" * 60)
print("CHECK 5: APGD accuracy decreases with increasing step count")
print("=" * 60)

EPS = 8 / 255
step_counts = [1, 5, 10, 20]
accuracies = []

for steps in step_counts:
    apgd = APGDPerturbation(epsilon=EPS, num_steps=steps, num_restarts=1)
    with torch.enable_grad():
        adv = apgd.apply(model, images_norm, labels, device)
    with torch.no_grad():
        preds = model(adv).argmax(dim=1)
    acc = (preds == labels).float().mean().item()
    accuracies.append(acc)
    print(f"  steps={steps:>3}: accuracy = {acc:.4f}")

# Check monotonically non-increasing
is_non_increasing = all(accuracies[i] >= accuracies[i+1] for i in range(len(accuracies)-1))
status = "PASS" if is_non_increasing else "FAIL (non-monotonic — may be acceptable due to randomness)"
print(f"\n  Monotonically non-increasing: [{status}]")

plt.figure(figsize=(5, 3))
plt.plot(step_counts, accuracies, marker='o')
plt.xlabel("APGD steps")
plt.ylabel("Accuracy")
plt.title("APGD accuracy vs step count (ResNet-18 CIFAR-10, ε=8/255)")
plt.tight_layout()
plt.show()

## Check 6 — Pipeline ordering: Compression→Attack ≠ Attack→Compression

Expected: the two pipelines produce different outputs, confirming order is correctly applied.

In [ ]:
print("=" * 60)
print("CHECK 6: Compression→Attack ≠ Attack→Compression")
print("=" * 60)

EPS = 8 / 255
Q   = 50

for comp_name, comp_cls in [("JPEG",     JpegPerturbation(quality=Q)),
                             ("PCA",      PcaPerturbation(quality=Q)),
                             ("PatchSVD", PatchSVDPerturbation(quality=Q))]:
    fgsm = FgsmPerturbation(epsilon=EPS)

    pipe_ca = PerturbationPipeline(name="comp->atk", steps=[comp_cls, FgsmPerturbation(epsilon=EPS)])
    pipe_ac = PerturbationPipeline(name="atk->comp", steps=[FgsmPerturbation(epsilon=EPS), comp_cls])

    with torch.enable_grad():
        out_ca = pipe_ca.apply(model, images_norm, labels, device)
        out_ac = pipe_ac.apply(model, images_norm, labels, device)

    max_diff = (out_ca - out_ac).abs().max().item()
    status = "PASS" if max_diff > 1e-6 else "FAIL (outputs identical — ordering not applied)"
    print(f"  {comp_name}: max diff between orderings = {max_diff:.4f}  [{status}]")

## Check 7 — PSNR manual computation vs framework output

Expected: manual PSNR matches `batch_metrics_from_normalized` to within floating-point precision.

In [ ]:
print("=" * 60)
print("CHECK 7: Manual PSNR vs batch_metrics_from_normalized")
print("=" * 60)

# Apply JPEG q=50 compression to get a perturbed batch
pert = JpegPerturbation(quality=50)
out = pert.apply(None, images_norm, labels, device)

# Framework computation
mse_fw, _, psnr_fw = batch_metrics_from_normalized(images_norm, out)

# Manual computation on first 4 images
x0 = denormalize(images_norm[:4])
x1 = denormalize(out[:4])

manual_psnr_values = []
for i in range(4):
    mse_i = ((x0[i] - x1[i]) ** 2).mean().item()
    psnr_i = 10.0 * math.log10(1.0 / max(mse_i, 1e-10))
    manual_psnr_values.append(psnr_i)

print(f"  {'Image':<8} {'Framework PSNR':>18} {'Manual PSNR':>14} {'Diff':>10} {'Status':>8}")
print(f"  {'-'*60}")
all_pass = True
for i in range(4):
    fw  = psnr_fw[i].item()
    man = manual_psnr_values[i]
    diff = abs(fw - man)
    ok = diff < 1e-3
    all_pass = all_pass and ok
    print(f"  {i:<8} {fw:>18.4f} {man:>14.4f} {diff:>10.2e} {'PASS' if ok else 'FAIL':>8}")
print(f"\n  Overall: [{'PASS' if all_pass else 'FAIL'}]")

## Check 8 — Clean accuracy vs published baselines

Expected published baselines (timm pretrained):
- `resnet18_cifar10` ≈ 93–95%
- `resnet50_cifar10` ≈ 95–96%

In [ ]:
print("=" * 60)
print("CHECK 8: Clean accuracy vs published baselines (full test set)")
print("=" * 60)

BASELINES = {
    "resnet18_cifar10": (0.93, 0.96),
    "resnet50_cifar10": (0.94, 0.97),
}

full_loader = DataLoader(cifar10_test, batch_size=256, shuffle=False, num_workers=2)

for model_name, (lo, hi) in BASELINES.items():
    m = load_model(model_name)
    correct = total = 0
    with torch.no_grad():
        for imgs, lbls in full_loader:
            preds = m(imgs.to(device)).argmax(dim=1)
            correct += (preds == lbls.to(device)).sum().item()
            total   += lbls.size(0)
    acc = correct / total
    status = "PASS" if lo <= acc <= hi else "FAIL"
    print(f"  {model_name}: accuracy = {acc:.4f}  (expected {lo:.2f}–{hi:.2f})  [{status}]")

## Check 9 — LIC-ROI visual inspection (ImageNet only)

This check requires the CompressAI model and an ImageNet image.
It is skipped if CompressAI or a sample image is unavailable.

The check visualises: original | LQ reconstruction | HQ reconstruction | blended ROI output,
with the saliency mask overlaid, to confirm salient regions receive higher-quality reconstruction.

In [ ]:
try:
    from compressai.zoo import cheng2020_attn
    _compressai_available = True
except ImportError:
    _compressai_available = False
    print("CompressAI not available — skipping LIC-ROI visual check.")

# To run this check, set IMAGENET_SAMPLE to the path of a 224×224 ImageNet image.
IMAGENET_SAMPLE = None  # e.g. "/data/shared/imagenet/val/n01440764/ILSVRC2012_val_00000293.JPEG"

if _compressai_available and IMAGENET_SAMPLE is not None:
    IN_MEAN = (0.485, 0.456, 0.406)
    IN_STD  = (0.229, 0.224, 0.225)
    IN_MEAN_T = torch.tensor(IN_MEAN, device=device).view(1, 3, 1, 1)
    IN_STD_T  = torch.tensor(IN_STD,  device=device).view(1, 3, 1, 1)

    def in_denorm(x):
        return torch.clamp(x * IN_STD_T + IN_MEAN_T, 0.0, 1.0)

    def in_renorm(x):
        return (x - IN_MEAN_T) / IN_STD_T

    # Load sample image
    tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(IN_MEAN, IN_STD)])
    img_pil = Image.open(IMAGENET_SAMPLE).convert("RGB")
    img_norm = tf(img_pil).unsqueeze(0).to(device)

    # Load ResNet-50 ImageNet
    import torchvision.models as tvm
    rn50 = tvm.resnet50(weights=tvm.ResNet50_Weights.DEFAULT).to(device).eval()

    # Saliency mask
    def get_saliency_map(x_norm, model):
        x = x_norm.detach().requires_grad_(True)
        logits = model(x)
        loss = logits[0, logits.argmax()].sum()
        grad = torch.autograd.grad(loss, x)[0]
        sal = grad.abs().mean(dim=1, keepdim=True)
        sal = (sal - sal.min()) / (sal.max() - sal.min() + 1e-8)
        return sal.detach()

    # CompressAI compression at two quality levels
    _cache = {}
    def compress_decompress(x_px, level):
        if level not in _cache:
            net = cheng2020_attn(quality=level, pretrained=True).to(device).eval()
            net.update()
            _cache[level] = net
        with torch.no_grad():
            return _cache[level](x_px)["x_hat"].clamp(0.0, 1.0)

    x_px = in_denorm(img_norm)
    H, W = x_px.shape[2], x_px.shape[3]
    pad_h = (64 - H % 64) % 64
    pad_w = (64 - W % 64) % 64
    x_pad = F.pad(x_px, (0, pad_w, 0, pad_h), mode="reflect")

    hq = compress_decompress(x_pad, level=4)[:, :, :H, :W]
    lq = compress_decompress(x_pad, level=2)[:, :, :H, :W]

    mask = get_saliency_map(img_norm, rn50)
    blended = mask * hq + (1 - mask) * lq

    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    titles = ["Original", "LQ (level 2)", "HQ (level 4)", "Saliency mask", "Blended ROI"]
    imgs   = [x_px, lq, hq, mask.expand_as(x_px), blended]
    for ax, title, im in zip(axes, titles, imgs):
        ax.imshow(im[0].permute(1, 2, 0).cpu().clamp(0, 1).numpy())
        ax.set_title(title)
        ax.axis("off")
    plt.suptitle("CHECK 9: LIC-ROI visual inspection", y=1.02)
    plt.tight_layout()
    plt.show()
    print("Visual inspection: confirm salient regions (bright mask areas) show higher sharpness in blended output.")
else:
    print("Skipped: set IMAGENET_SAMPLE to a valid ImageNet image path and ensure CompressAI is installed.")

---
## Summary

All automated checks above print `[PASS]` or `[FAIL]`. A successful run confirms:

| # | Check | Expected result |
|---|-------|-----------------|
| 1a | JPEG at q=100 | PSNR > 35 dB (lossy by design) |
| 1b | JPEG2000 at q=100 | PSNR > 50 dB (near-lossless wavelet) |
| 2 | PCA / PatchSVD at q=100 | PSNR > 60 dB, MSE ≈ 0 |
| 3 | Attacks at ε=0 | MSE < 1e-8 |
| 4 | FGSM == PGD(k=1, no random start) | Max diff < 1e-6 |
| 5 | APGD accuracy vs step count | Monotonically non-increasing |
| 6 | Compression→Attack ≠ Attack→Compression | Max diff > 1e-6 |
| 7 | Manual PSNR vs framework PSNR | Diff < 1e-3 dB |
| 8 | Clean accuracy vs published baselines | Within expected range |
| 9 | LIC-ROI visual inspection (ImageNet) | Manual / visual |